# Load Aviva Resource Tracker

This notebook locates the Excel file in the `inputs` folder (either `resource.xlsx` or `Aviva Resource Tracker_May 2025.xlsm`) using absolute paths to prevent any working directory resolution errors, and loads the data into a pandas DataFrame.

In [2]:
import os
import pandas as pd
import warnings

# Suppress openpyxl conditional formatting warnings
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [3]:
def get_excel_path(project_root, spreadsheet):
    possible_paths = [
        os.path.join(project_root, 'inputs', spreadsheet)
    ]   
    print(possible_paths)
    excel_path = None
    for path in possible_paths:
        if os.path.exists(path):
            excel_path = path
            break

    if excel_path is None:
        raise FileNotFoundError(
            f"Could not find any resource tracker Excel file in the inputs directory.\n"
            f"Looked for:\n" + "\n".join(possible_paths)
        )

    return excel_path

In [4]:
# Define the absolute project root directory
project_root = "/Users/MMIE/projects/insurance_dashboard"

# Check if the current working directory is stale and recover it if needed
try:
    cwd = os.getcwd()
    print(f"Current working directory: {cwd}")
except FileNotFoundError:
    print("WARNING: The Jupyter kernel's working directory is stale (likely because a directory was deleted/recreated).")
    if os.path.exists(project_root):
        os.chdir(project_root)
        print(f"Successfully recovered working directory by switching to project root: {os.getcwd()}")
    else:
        print("Could not find the project root directory. Please check the project path.")

# Define potential absolute paths to the Excel files

excel_path = get_excel_path(project_root, "resource.xlsx")

print(f"Using Excel file: {excel_path}")

Current working directory: /Users/MMIE/projects/insurance_dashboard/notebooks
['/Users/MMIE/projects/insurance_dashboard/inputs/resource.xlsx']
Using Excel file: /Users/MMIE/projects/insurance_dashboard/inputs/resource.xlsx


In [12]:
def get_target_sheet_names(excel_path):
    # Load available sheet names to determine the correct target sheet
    xls = pd.ExcelFile(excel_path)
    sheet_names = xls.sheet_names
    print(f"Available sheets in file: {sheet_names}")

    # Determine sheet to load ('Change Sourcing' if present, otherwise fallback to 'Sheet1' or the first sheet)
    target_sheet = 'Change Sourcing'
    if target_sheet not in sheet_names:
        if 'Sheet1' in sheet_names:
            target_sheet = 'Sheet1'
        else:
            target_sheet = sheet_names[0]

    print(f"Target sheet to load: '{target_sheet}'")
    return target_sheet


In [6]:
target_sheet = get_target_sheet_names(excel_path)


Available sheets in file: ['Sheet1']
Target sheet to load: 'Sheet1'


In [7]:
# Load the sheet into a pandas DataFrame
df = pd.read_excel(excel_path, sheet_name=target_sheet)

print(f"Successfully loaded data.")
print(f"DataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns.")

Successfully loaded data.
DataFrame shape: 1310 rows, 40 columns.


In [8]:
# Preview the loaded data
df_cleansed = df.dropna(subset=["Actual Start Date"])
df_cleansed.head()

,Demand on SDCPOT Portal (Yes/No),GETGO Reference Number,Capco Role Reference Number,Details,Aviva Business Area,Aviva Sub Business Area,Role Title\n(Rate Card),Role Level\n(Rate Card),Specialism\n(Rate Card),Specialist Skills,...,Status (KPI).1,Date Interview Feedback Received,Resource Confirmed (Yes/No),Resource Location (Capco office),Target Start Date,"Contracting Status\n(PO issued, Pending, Not Requested)",Actual Start Date,Resource Roll Off Date,Additional Comments,Unnamed: 39
7,YES,1029,93663,Senior Business Analysis - Hybrid BA-PM - RMP ...,Life & Pension,New Consumer Duty,Business Analyst,Senior,Generic,Skills Required: Dimensions:\n- Gather and cla...,...,Green,2026-01-30 00:00:00,NaN,NaN,2026-02-12 00:00:00,Issued,2026-02-12 00:00:00,NaN,NaN,NaN
8,NO,TBC,69024,"Senior Business Analysis - Product feature, Ag...",Life & Pension,IWR CIO,Business Analyst,Senior,Generic,Agile delivery experience (definite)\nAbility ...,...,Green,2024-10-31 00:00:00,Yes,London,2024-11-27 00:00:00,Issued,2024-11-27 00:00:00,2026-04-30 00:00:00,NaN,NaN
10,NO,NaN,29214,Aviva Pipeline - Agile Scrum Master,Tech Change Delivery,C&M,Scrum Master,Intermediate,Generic,Scrum Master,...,Green,2022-07-25 00:00:00,Yes,NaN,2022-08-01 00:00:00,Issued,2022-08-01 00:00:00,2022-08-23 00:00:00,NaN,NaN
11,NO,NaN,30419,Aviva Pipeline - Solution Architect,Solution Architecture,NaN,Solution Architect,Intermediate,Generic,NaN,...,Green,2022-08-02 00:00:00,Yes,NaN,2022-08-18 00:00:00,Issued,2022-08-18 00:00:00,2023-04-19 00:00:00,NaN,NaN
18,NO,NaN,30438,Aviva Pipeline - PMO,PMO,NaN,Project Manager,Intermediate,Generic,NaN,...,Green,2022-07-21 00:00:00,Yes,NaN,2022-08-08 00:00:00,Issued,2022-08-08 00:00:00,2022-10-21 00:00:00,Performance issue noted by client - backfill r...,NaN


In [11]:
# Convert to datetime to allow date operations
df_cleansed["Actual Start Date"] = pd.to_datetime(df_cleansed["Actual Start Date"], errors="coerce")
df_cleansed["Resource Roll Off Date"] = pd.to_datetime(df_cleansed["Resource Roll Off Date"], errors="coerce")

# Calculate the previous calendar month dynamically
prev_month = pd.Timestamp.now() - pd.DateOffset(months=1)

new_starters_df = df_cleansed[
    (df_cleansed["Status"].str.contains("Started at Aviva", na=False)) &
    (df_cleansed["Actual Start Date"].dt.year == prev_month.year) &
    (df_cleansed["Actual Start Date"].dt.month == prev_month.month)
]

rolled_off_df = df_cleansed[
    (df_cleansed["Status"].str.contains("Rolled off", na=False, case=False)) &
    (df_cleansed["Resource Roll Off Date"].dt.year == prev_month.year) &
    (df_cleansed["Resource Roll Off Date"].dt.month == prev_month.month)
]

#print(new_starters_df["Candidate Name"])
print(rolled_off_df["Candidate Name"])


Series([], Name: Candidate Name, dtype: object)


/var/folders/bc/42kgzvb144nf6h35hj2_05wr0000gp/T/ipykernel_21204/478235368.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleansed["Actual Start Date"] = pd.to_datetime(df_cleansed["Actual Start Date"], errors="coerce")
/var/folders/bc/42kgzvb144nf6h35hj2_05wr0000gp/T/ipykernel_21204/478235368.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleansed["Resource Roll Off Date"] = pd.to_datetime(df_cleansed["Resource Roll Off Date"], errors="coerce")


In [ ]:
excel_path = get_excel_path(project_root, "dashboard.xlsx")
print(excel_path)
target_sheet = get_target_sheet_names(excel_path)

['/Users/MMIE/projects/insurance_dashboard/inputs/dashboard.xlsx']
/Users/MMIE/projects/insurance_dashboard/inputs/dashboard.xlsx
Available sheets in file: ['Sheet1']
Target sheet to load: 'Sheet1'


In [ ]:
# Load dashboard source data

df_dashboard = pd.read_excel(excel_path, sheet_name=target_sheet)

print(f"Successfully loaded data.")
print(f"DataFrame shape: {df.shape[0]} rows, {df.shape[1]} columns.")

df_dashboard

Successfully loaded data.
DataFrame shape: 1310 rows, 40 columns.


,Resource Full Name,Business Unit,Project / Programme,Project Type,Change Sourcing,Active,Summary of Project,Capco Role,Exec Sponsor,Aviva Key Stakeholders,Capco Line Manager,Functional Leads,Capco Location City,Capco Location Country,Aviva Role Location,Last Updated,Technology Platform / Systems (if applicable),Credentials Created,Credentials received in Oct 2025
0,Ashish Bhatt,General Insurance,1. Reinsurance Transformation\n2. Tax Pillar 2...,Strategic project,Yes,Yes,1. Reinsurance Transformation across UK & Irel...,Solution Architect,1. Peter Taylor\n2. Neal Robinson\n3. C.J Bonham,Alicia Jennings,Somnath Ghosh,NaN,London,UK,London,17.1.25,"1. DXC, Oracle, GW, SEND, AWS\n2. Pillar 2 Age...",No,Chased 07/11/2025
1,Gajendran Sundaramurthy,"Insurance, Wealth & Retirement",2024 Data Deletion Programme,Regulatory change,Yes,Yes,Aviva had initiated an original end to end GDP...,Programme Manager,Neil Grady,"Oliver Long - Ops Sponsor, Mel Spencer - Data ...",Arun Vasan,NaN,Bangalore,India,India,21.1.25,"FNZ, Phoenix, Alfresco, Roxi, Alps lite",Yes,NaN
2,Bhavya Shingari,"Insurance, Wealth & Retirement",Aviva Save,Digital transformation,Yes,Yes,Aviva Save project is re-platforming of Aviva'...,Business Analyst,Clyde Raghoobur,"PO: Lee Coolahan, PM: Michael Willis",Zoheb Ahmed,NaN,London,UK,London,29.10.25,NaN,Yes,Yes
3,Rohan Sarode,"Insurance, Wealth & Retirement",AVIVA Sustainability,Strategic project,Yes,Yes,The Aviva IWR ASA program delivery will provid...,Project Manager,Peter Jones,"Ashish Dafria, CIO, Bianca Hanscombe - Head of...",Santhosh Vaniamparambath,NaN,Pune,India,London,17.1.25,NaN,No,Yes (13/10)
4,Manish Bhagat,Cross-Group functions,AWS Gen 2 and Ongoing Maintenance,Cost saving,Yes,Yes,AWS Gen 2-Aviva is seeking to improve the cust...,PMO,"John Lyon, Lee Ann","Matthew Bird, Lee Macey, Neil Priestland, Emil...",Ravikumar Ganne,NaN,Bangalore,India,Norwich,21.1.25,"Cloud DBT, Snowflake, Collibra, Cloud,Platforms",Yes,Yes (13/10)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,Kimberlee McLaughlan,Cross-Group functions,Multinational Project - formally part of Nova ...,Strategic project,Yes,Yes,Technology - standing up insurance workflow pl...,Project Manager,Belinda Steedman,"Matthew Field, Matthew Read",Gillian Spanswick,Edinburgh,Edinburgh,UK,Perth,2026-06-12 00:00:00,iCede,NaN,NaN
131,Kenny McGavin,General Insurance,River MADI Programme (Marketing Analytics and ...,Data migration,Yes,Yes,The Marketing Analytics & Data Integration Pro...,Business Analyst,"Operational sponsor - Nitin Madelil, Exec Spon...","Steph Ismail, Fiona Park, Joe Corbani, Isobel ...",Maria Archana,Edinburgh,Edinburgh,UK,Perth,2026-06-12 00:00:00,"SCM, Cyclops, Snowflake, Distribution Hub, PAS",NaN,NaN
132,Luc Viergever,NaN,NaN,NaN,Yes,Yes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
133,Shriya Kotecha,Cross-Group functions,Adobe Workfront & DAM,Digital transformation,Yes,Yes,Part of the Customer and Marketing team to mov...,Business Analyst,Claire Davies,"Crystal Graham, Ashutosh Singh, Jordan Rant, N...",Sonal Modi,London,London,UK,London,2026-06-12 00:00:00,Adobe Workfront and DAM\nDocumentum\nSmartshee...,NaN,NaN


In [ ]:
# remove duplicates from new starters dataframe 
duplicates = []
for name in new_starters_df["Candidate Name"]:
    if name in df_dashboard["Resource Full Name"].values:
        duplicates.append(name)
# Remove duplicates from new_starters_df
new_starters_df = new_starters_df[~new_starters_df["Candidate Name"].isin(duplicates)]
print(f"Removed duplicate candidates: {duplicates}")
print(new_starters_df.columns)


Removed duplicate candidates: ['Lee Reilly']
Index(['Demand on SDCPOT Portal (Yes/No)', 'GETGO Reference Number',
       'Capco Role Reference Number', 'Details', 'Aviva Business Area',
       'Aviva Sub Business Area', 'Role Title\n(Rate Card)',
       'Role Level\n(Rate Card)', 'Specialism\n(Rate Card)',
       'Specialist Skills', 'Onshore/Offshore', 'Hiring Manager',
       'Aviva Resource Manager', 'Capco Lead', 'Status', 'Candidate Name',
       'Grade', 'Date Raised\n(SLA)', 'Date Qualified\n(SLA)',
       'Date CV Sent\n(SLA)',
       'Business Days (SLA calculation, from Date Raised)', 'Status (SLA) 1',
       'Business Days (SLA calculation, from Date Qualified)',
       'Status (SLA) 2', 'Date CV Feedback Received (KPI)',
       'Date Interview Scheduled\n(KPI)', 'Business Days (KPI calculation)',
       'Status (KPI)', 'Interview Date\n(KPI)',
       'Business Days (KPI calculation).1', 'Status (KPI).1',
       'Date Interview Feedback Received', 'Resource Confirmed (Yes/No

In [ ]:
# Go through the rolled_off_df dataframe and search df_dashboard for rolled-off candidates by name
rolled_off_names = rolled_off_df["Candidate Name"].tolist()

# Remove matching rows from df_dashboard
df_dashboard = df_dashboard[~df_dashboard["Resource Full Name"].isin(rolled_off_names)]

print(f"Removed rolled-off candidates from dashboard: {rolled_off_names}")
print(f"New df_dashboard shape: {df_dashboard.shape[0]} rows, {df_dashboard.shape[1]} columns.")

print(df_dashboard.columns)


Removed rolled-off candidates from dashboard: []
New df_dashboard shape: 135 rows, 19 columns.
Index(['Resource Full Name', 'Business Unit ', 'Project / Programme',
       'Project Type', 'Change Sourcing', 'Active', 'Summary of Project ',
       'Capco Role', 'Exec Sponsor', 'Aviva Key Stakeholders',
       'Capco Line Manager ', 'Functional Leads', 'Capco Location City',
       'Capco Location Country', 'Aviva Role Location', 'Last Updated',
       'Technology Platform / Systems (if applicable)', 'Credentials Created',
       'Credentials received in Oct 2025'],
      dtype='str')


In [ ]:
# Create a DataFrame for the new starters with columns matching df_dashboard
new_rows = []

# Get mode values of df_dashboard for fill/inference where needed
modes = {col: df_dashboard[col].mode()[0] if not df_dashboard[col].mode().empty else None for col in df_dashboard.columns}

for _, row in new_starters_df.iterrows():
    new_row = {}
    
    # 1. Map columns from new_starters_df
    new_row["Resource Full Name"] = row["Candidate Name"]
    new_row["Business Unit "] = row["Aviva Business Area"]
    new_row["Capco Role"] = row["Role Title\n(Rate Card)"]
    new_row["Capco Location City"] = row["Resource Location (Capco office)"]
    new_row["Last Updated"] = row["Actual Start Date"]
    
    # 2. Infer Capco Location Country based on city name
    city = str(row["Resource Location (Capco office)"]).strip().lower() if pd.notna(row["Resource Location (Capco office)"]) else ""
    if any(c in city for c in ["london", "edinburgh", "manchester", "perth", "york"]):
        new_row["Capco Location Country"] = "UK"
    elif any(c in city for c in ["bangalore", "bengaluru", "pune", "gurgaon", "gurugram", "mumbai", "hyderabad"]):
        new_row["Capco Location Country"] = "India"
    else:
        new_row["Capco Location Country"] = modes["Capco Location Country"]
        
    # 3. Apply standard defaults
    new_row["Active"] = "Yes"
    new_row["Credentials Created"] = "No"
    new_row["Credentials received in Oct 2025"] = None
    new_row["Functional Leads"] = None
    
    # 4. Fill remaining fields using the mode values from df_dashboard
    for col in df_dashboard.columns:
        if col not in new_row:
            new_row[col] = modes[col]
            
    new_rows.append(new_row)

new_starters_dashboard_df = pd.DataFrame(new_rows)

# Add new starters to df_dashboard
df_dashboard = pd.concat([df_dashboard, new_starters_dashboard_df], ignore_index=True)

print(f"Successfully added {len(new_starters_dashboard_df)} new starters to df_dashboard.")
print(f"New Starters")
print(new_starters_dashboard_df["Resource Full Name"].to_string(index=False))

Successfully added 0 new starters to df_dashboard.
New Starters


KeyError: 'Resource Full Name'

In [ ]:
print(df_dashboard["Resource Full Name"])


0                 Ashish Bhatt
1      Gajendran Sundaramurthy
2              Bhavya Shingari
3                 Rohan Sarode
4                Manish Bhagat
                ...           
130       Kimberlee McLaughlan
131              Kenny McGavin
132              Luc Viergever
133             Shriya Kotecha
134              Charlotte Lee
Name: Resource Full Name, Length: 135, dtype: str
